# Deploy Microsoft Fabric ETL Migration Accelerator (FEMA)

Run this notebook inside your target Microsoft Fabric workspace (with a default Lakehouse attached for the staged `fema-deployment-bundle` files) to provision or upgrade:

1. Assessment Lakehouse
2. Fabric Runtime 2.0 Environment with the accelerator wheel
3. Portfolio Assessment Notebook bound to Lakehouse and Environment
4. Initialized empty `etl_*` Delta tables (idempotent schema-on-first-deploy)
5. Direct Lake Semantic Model bound to Workspace and Lakehouse
6. Six-page Power BI Report bound to the Semantic Model

Existing Lakehouse Delta tables and retained assessment runs are preserved on upgrades.

In [ ]:
from pathlib import Path

BUNDLE_DIR = Path('/lakehouse/default/Files/fema-deployment-bundle')
FOLDER_NAME = 'FEMA'
LAKEHOUSE_NAME = 'ETL_Migration_Assessment_Lakehouse'
ENVIRONMENT_NAME = 'ETL_Migration_Assessment_Env'
NOTEBOOK_NAME = 'ETL Migration Assessment'
SEMANTIC_MODEL_NAME = 'ETL Migration Assessment Model'
REPORT_NAME = 'ETL Migration Assessment'
PUBLISH_ENVIRONMENT = True
BOOTSTRAP_DELTA_SCHEMA = True
CONFIRM_DEPLOY = False  # Set True after verifying BUNDLE_DIR files.


In [ ]:
import base64
import json
import time
import urllib.error
import urllib.request
import uuid

FABRIC_API = 'https://api.fabric.microsoft.com/v1'


def get_fabric_token(resource: str = 'https://api.fabric.microsoft.com') -> str:
    if 'notebookutils' in globals():
        return notebookutils.credentials.getToken(resource)
    if 'mssparkutils' in globals():
        return mssparkutils.credentials.getToken(resource)
    raise RuntimeError('Run this notebook inside Microsoft Fabric to acquire a Fabric token.')


def current_workspace_id() -> str:
    if 'fabric' in globals() and hasattr(fabric, 'get_workspace_id'):
        return str(fabric.get_workspace_id())
    try:
        import sempy.fabric as sempy_fabric
        return str(sempy_fabric.get_workspace_id())
    except Exception as exc:
        raise RuntimeError('Unable to detect the current Fabric workspace ID.') from exc


def fabric_request(method: str, url: str, token: str, body=None, raw_bytes: bytes | None = None, content_type: str | None = None, poll_lro: bool = True):
    headers = {
        'Authorization': f'Bearer {token}',
        'x-ms-fabric-skill': 'onelake-catalog-cli',
    }
    data = None
    if raw_bytes is not None:
        data = raw_bytes
        headers['Content-Type'] = content_type or 'application/octet-stream'
    elif body is not None:
        data = json.dumps(body).encode('utf-8')
        headers['Content-Type'] = 'application/json'
    elif method == 'POST':
        headers['Content-Length'] = '0'

    for attempt in range(6):
        req = urllib.request.Request(url, data=data, headers=headers, method=method)
        try:
            with urllib.request.urlopen(req) as resp:
                status = resp.status
                resp_headers = dict(resp.headers)
                text = resp.read().decode('utf-8')
        except urllib.error.HTTPError as err:
            if err.code in (429, 503) and attempt < 5:
                delay = int(err.headers.get('Retry-After', '5') or '5')
                time.sleep(delay)
                continue
            detail = err.read().decode('utf-8', errors='replace')
            raise RuntimeError(f'Fabric API {method} {url} failed ({err.code}): {detail}') from err

        if status == 202 and poll_lro:
            op_url = resp_headers.get('Location') or f"{FABRIC_API}/operations/{resp_headers.get('x-ms-operation-id', '')}"
            retry_after = int(resp_headers.get('Retry-After', '5') or '5')
            for _ in range(120):
                time.sleep(retry_after)
                op = fabric_request('GET', op_url, token)
                op_status = (op or {}).get('status')
                if op_status in ('Succeeded', 'Completed'):
                    return op
                if op_status in ('Failed', 'Cancelled', 'Deduped'):
                    raise RuntimeError(f'Fabric operation failed ({op_status}): {op}')
            raise TimeoutError(f'Timed out polling Fabric operation: {op_url}')

        return json.loads(text) if text.strip() else None


def list_items(workspace_id: str, collection: str, token: str) -> list[dict]:
    items = []
    next_url = f'{FABRIC_API}/workspaces/{workspace_id}/{collection}'
    while next_url:
        page = fabric_request('GET', next_url, token) or {}
        items.extend(page.get('value', []))
        next_url = page.get('continuationUri') or ''
    return items



def resolve_or_create_folder(workspace_id: str, folder_path: str, token: str) -> str | None:
    segments = [seg.strip() for seg in folder_path.replace('\\', '/').split('/') if seg.strip()]
    if not segments:
        return None
    folders = list_items(workspace_id, 'folders', token)
    parent_id = None
    for segment in segments:
        matches = [
            f for f in folders
            if f.get('displayName') == segment and (
                (parent_id is None and not f.get('parentFolderId'))
                or f.get('parentFolderId') == parent_id
            )
        ]
        if len(matches) > 1:
            raise RuntimeError(f'Multiple folders matched {segment!r}.')
        if matches:
            parent_id = matches[0]['id']
            continue
        body = {'displayName': segment}
        if parent_id is not None:
            body['parentFolderId'] = parent_id
        created = fabric_request('POST', f'{FABRIC_API}/workspaces/{workspace_id}/folders', token, body=body)
        parent_id = created['id']
        folders.append(created)
    return parent_id


def move_item_to_folder(workspace_id: str, item_id: str, current_folder_id: str | None, target_folder_id: str | None, token: str) -> None:
    if not target_folder_id or current_folder_id == target_folder_id:
        return
    fabric_request(
        'POST',
        f'{FABRIC_API}/workspaces/{workspace_id}/items/{item_id}/move',
        token,
        body={'targetFolderId': target_folder_id},
    )


def resolve_or_create(workspace_id: str, collection: str, display_name: str, description: str, token: str, folder_id: str | None = None) -> dict:
    matches = [item for item in list_items(workspace_id, collection, token) if item.get('displayName') == display_name]
    if len(matches) > 1:
        raise RuntimeError(f'Multiple {collection} items matched {display_name!r}.')
    if matches:
        move_item_to_folder(workspace_id, matches[0]['id'], matches[0].get('folderId'), folder_id, token)
        return matches[0]
    create_body = {
        'displayName': display_name,
        'description': description,
    }
    if folder_id:
        create_body['folderId'] = folder_id
    created = fabric_request('POST', f'{FABRIC_API}/workspaces/{workspace_id}/{collection}', token, body=create_body)
    if created and created.get('id'):
        return created
    refetched = [item for item in list_items(workspace_id, collection, token) if item.get('displayName') == display_name]
    if len(refetched) != 1:
        raise RuntimeError(f'Created {collection} item {display_name!r} could not be resolved.')
    return refetched[0]


def upsert_defined_item(workspace_id: str, collection: str, display_name: str, payload: dict, token: str, folder_id: str | None = None) -> dict:
    matches = [item for item in list_items(workspace_id, collection, token) if item.get('displayName') == display_name]
    if len(matches) > 1:
        raise RuntimeError(f'Multiple {collection} items matched {display_name!r}.')
    if matches:
        item_id = matches[0]['id']
        move_item_to_folder(workspace_id, item_id, matches[0].get('folderId'), folder_id, token)
        has_platform = any(part.get('path') == '.platform' for part in payload['definition']['parts'])
        suffix = '?updateMetadata=true' if has_platform else ''
        fabric_request(
            'POST',
            f'{FABRIC_API}/workspaces/{workspace_id}/{collection}/{item_id}/updateDefinition{suffix}',
            token,
            body={'definition': payload['definition']},
        )
        return matches[0]
    create_payload = {**payload, 'folderId': folder_id} if folder_id else payload
    created = fabric_request('POST', f'{FABRIC_API}/workspaces/{workspace_id}/{collection}', token, body=create_payload)
    if created and created.get('id'):
        return created
    refetched = [item for item in list_items(workspace_id, collection, token) if item.get('displayName') == display_name]
    if len(refetched) != 1:
        raise RuntimeError(f'Created {collection} item {display_name!r} could not be resolved.')
    return refetched[0]


def upload_multipart_file(url: str, file_path: Path, token: str) -> None:
    boundary = f'----FabricBoundary{uuid.uuid4().hex}'
    file_bytes = file_path.read_bytes()
    body = (
        f'--{boundary}\r\n'
        f'Content-Disposition: form-data; name="file"; filename="{file_path.name}"\r\n'
        'Content-Type: application/octet-stream\r\n\r\n'
    ).encode('utf-8') + file_bytes + f'\r\n--{boundary}--\r\n'.encode('utf-8')
    fabric_request('POST', url, token, raw_bytes=body, content_type=f'multipart/form-data; boundary={boundary}')


In [ ]:
import re
if not CONFIRM_DEPLOY:
    raise RuntimeError('Stage fema-deployment-bundle in Lakehouse Files and set CONFIRM_DEPLOY = True to run deployment.')
if not BUNDLE_DIR.is_dir():
    raise FileNotFoundError(f'Deployment bundle directory not found: {BUNDLE_DIR}')

wheel_dirs = [BUNDLE_DIR / 'dist', BUNDLE_DIR / 'wheels']
wheel_patterns = ('etl_migration_core-*.whl', 'etl_portfolio_assessment-*.whl', 'etl_target_fabric-*.whl')
wheel_paths = []
missing_patterns = []
for pattern in wheel_patterns:
    found = sorted(p for d in wheel_dirs if d.is_dir() for p in d.glob(pattern))
    if found:
        wheel_paths.append(found[-1])
    else:
        missing_patterns.append(pattern)
if missing_patterns:
    raise FileNotFoundError(f'Missing required .whl file(s) under BUNDLE_DIR/dist or BUNDLE_DIR/wheels: {missing_patterns}')
env_yml = BUNDLE_DIR / 'scripts' / 'fabric-environment.yml'
nb_source = BUNDLE_DIR / 'notebooks' / 'fabric-portfolio-assessment.ipynb'
sm_dir = BUNDLE_DIR / 'powerbi' / 'ETL Migration Assessment Model.SemanticModel'
rpt_dir = BUNDLE_DIR / 'powerbi' / 'ETL Migration Assessment.Report'

token = get_fabric_token()
workspace_id = current_workspace_id()
print(f'Target workspace: {workspace_id}')

folder_id = resolve_or_create_folder(workspace_id, FOLDER_NAME, token) if FOLDER_NAME else None
if folder_id:
    print(f'Folder ready: {FOLDER_NAME} ({folder_id})')

# 1. Lakehouse
lakehouse = resolve_or_create(
    workspace_id, 'lakehouses', LAKEHOUSE_NAME,
    'Governed Lakehouse storing staged ETL intake files and normalized assessment Delta tables.', token, folder_id=folder_id,
)
lakehouse_id = lakehouse['id']
print(f'Lakehouse ready: {LAKEHOUSE_NAME} ({lakehouse_id})')

# 2. Environment
environment = resolve_or_create(
    workspace_id, 'environments', ENVIRONMENT_NAME,
    'Fabric Runtime 2.0 (Python 3.13) environment with ETL Migration Accelerator wheel.', token, folder_id=folder_id,
)
environment_id = environment['id']
if PUBLISH_ENVIRONMENT:
    while True:
        env_info = fabric_request('GET', f'{FABRIC_API}/workspaces/{workspace_id}/environments/{environment_id}', token)
        state = ((env_info or {}).get('properties') or {}).get('publishDetails', {}).get('state')
        if state not in ('Running', 'Waiting'):
            break
        time.sleep(5)
    fabric_request(
        'PATCH',
        f'{FABRIC_API}/workspaces/{workspace_id}/environments/{environment_id}/staging/sparkcompute?beta=false',
        token,
        body={'runtimeVersion': '2.0'},
    )
    import urllib.parse
    staged_libs = fabric_request('GET', f'{FABRIC_API}/workspaces/{workspace_id}/environments/{environment_id}/staging/libraries', token) or {}
    desired_names = {p.name for p in wheel_paths}
    for existing_wheel in ((staged_libs.get('customLibraries') or {}).get('wheelFiles') or []):
        if existing_wheel.startswith('fabric_etl_migration_accelerator-') or (
            existing_wheel.startswith(('etl_migration_core-', 'etl_portfolio_assessment-', 'etl_target_fabric-'))
            and existing_wheel not in desired_names
        ):
            encoded = urllib.parse.quote(existing_wheel, safe='')
            fabric_request('DELETE', f'{FABRIC_API}/workspaces/{workspace_id}/environments/{environment_id}/staging/libraries?libraryToDelete={encoded}', token)
    for wheel_path in wheel_paths:
        fabric_request(
            'POST',
            f'{FABRIC_API}/workspaces/{workspace_id}/environments/{environment_id}/staging/libraries/{wheel_path.name}',
            token,
            raw_bytes=wheel_path.read_bytes(),
            content_type='application/octet-stream',
        )
    fabric_request(
        'POST',
        f'{FABRIC_API}/workspaces/{workspace_id}/environments/{environment_id}/staging/libraries/importExternalLibraries',
        token,
        raw_bytes=env_yml.read_bytes(),
        content_type='application/octet-stream',
    )
    fabric_request('POST', f'{FABRIC_API}/workspaces/{workspace_id}/environments/{environment_id}/staging/publish?beta=false', token)
    while True:
        env_info = fabric_request('GET', f'{FABRIC_API}/workspaces/{workspace_id}/environments/{environment_id}', token)
        state = ((env_info or {}).get('properties') or {}).get('publishDetails', {}).get('state')
        if state not in ('Running', 'Waiting'):
            if state != 'Success':
                raise RuntimeError(f'Environment publish failed with state: {state}')
            break
        time.sleep(10)
print(f'Environment ready: {ENVIRONMENT_NAME} ({environment_id})')

# 3. Assessment Notebook
nb_doc = json.loads(nb_source.read_text(encoding='utf-8'))
nb_doc.setdefault('metadata', {})['dependencies'] = {
    'lakehouse': {
        'default_lakehouse': lakehouse_id,
        'default_lakehouse_name': LAKEHOUSE_NAME,
        'default_lakehouse_workspace_id': workspace_id,
        'known_lakehouses': [{'id': lakehouse_id}],
    },
    'environment': {
        'environmentId': environment_id,
        'workspaceId': workspace_id,
    },
}
nb_platform = {
    '$schema': 'https://developer.microsoft.com/json-schemas/fabric/gitIntegration/platformProperties/2.0.0/schema.json',
    'metadata': {'type': 'Notebook', 'displayName': NOTEBOOK_NAME},
    'config': {'version': '2.0', 'logicalId': '00000000-0000-0000-0000-000000000000'},
}
nb_payload = {
    'displayName': NOTEBOOK_NAME,
    'description': 'Deterministic portfolio assessment and normalized Lakehouse persistence.',
    'definition': {
        'format': 'ipynb',
        'parts': [
            {'path': 'notebook-content.ipynb', 'payload': base64.b64encode((json.dumps(nb_doc, indent=1) + '\n').encode('utf-8')).decode('ascii'), 'payloadType': 'InlineBase64'},
            {'path': '.platform', 'payload': base64.b64encode((json.dumps(nb_platform, indent=2) + '\n').encode('utf-8')).decode('ascii'), 'payloadType': 'InlineBase64'},
        ],
    },
}
notebook = upsert_defined_item(workspace_id, 'notebooks', NOTEBOOK_NAME, nb_payload, token, folder_id=folder_id)
notebook_id = notebook['id']
print(f'Notebook ready: {NOTEBOOK_NAME} ({notebook_id})')

# 4. Bootstrap Delta tables
if BOOTSTRAP_DELTA_SCHEMA:
    fabric_request(
        'POST',
        f'{FABRIC_API}/workspaces/{workspace_id}/notebooks/{notebook_id}/jobs/execute/instances?beta=false',
        token,
        body={'parameters': [
            {'name': 'INITIALIZE_SCHEMA_ONLY', 'value': True, 'type': 'Boolean'},
            {'name': 'PERSIST_DELTA_TABLES', 'value': False, 'type': 'Boolean'},
            {'name': 'CONFIRM_ASSESSMENT', 'value': False, 'type': 'Boolean'},
            {'name': 'WRITE_REPORTS', 'value': False, 'type': 'Boolean'},
        ]},
    )
    print('Delta schema bootstrap completed.')

# 5. Direct Lake Semantic Model
lh_details = fabric_request('GET', f'{FABRIC_API}/workspaces/{workspace_id}/lakehouses/{lakehouse_id}', token) or {}
default_schema = (lh_details.get('properties') or {}).get('defaultSchema') or ''
sm_parts = []
for file_path in sorted(p for p in sm_dir.rglob('*') if p.is_file()):
    rel = file_path.relative_to(sm_dir).as_posix()
    text = file_path.read_text(encoding='utf-8')
    text = text.replace('__FABRIC_WORKSPACE_ID__', workspace_id).replace('__FABRIC_LAKEHOUSE_ID__', lakehouse_id)
    if rel.startswith('definition/tables/') and rel.endswith('.tmdl'):
        if not default_schema:
            text = re.sub(r'^\t*schemaName:\s*\S+\r?\n', '', text, flags=re.MULTILINE)
        else:
            text = re.sub(r'^(\t*schemaName:\s*)\S+', lambda m: m.group(1) + default_schema, text, flags=re.MULTILINE)
    sm_parts.append({'path': rel, 'payload': base64.b64encode(text.encode('utf-8')).decode('ascii'), 'payloadType': 'InlineBase64'})
sm_payload = {
    'displayName': SEMANTIC_MODEL_NAME,
    'description': 'Direct Lake semantic model for governed ETL migration assessment analytics.',
    'definition': {'format': 'TMDL', 'parts': sm_parts},
}
semantic_model = upsert_defined_item(workspace_id, 'semanticModels', SEMANTIC_MODEL_NAME, sm_payload, token, folder_id=folder_id)
semantic_model_id = semantic_model['id']
print(f'Semantic Model ready: {SEMANTIC_MODEL_NAME} ({semantic_model_id})')

pbi_token = get_fabric_token('https://analysis.windows.net/powerbi/api')
fabric_request(
    'POST',
    f'https://api.powerbi.com/v1.0/myorg/groups/{workspace_id}/datasets/{semantic_model_id}/refreshes',
    pbi_token,
    body={'notifyOption': 'NoNotification'},
    poll_lro=False,
)
for _ in range(30):
    time.sleep(3)
    hist = fabric_request('GET', f'https://api.powerbi.com/v1.0/myorg/groups/{workspace_id}/datasets/{semantic_model_id}/refreshes?$top=1', pbi_token) or {}
    latest = (hist.get('value') or [None])[0]
    if not latest:
        continue
    if latest.get('status') == 'Completed':
        print('Semantic Model Direct Lake framing refresh completed.')
        break
    if latest.get('status') in ('Failed', 'Cancelled', 'Disabled'):
        raise RuntimeError(f'Semantic Model refresh failed: {latest}')

# 6. Power BI Report
rpt_parts = []
for file_path in sorted(p for p in rpt_dir.rglob('*') if p.is_file()):
    rel = file_path.relative_to(rpt_dir).as_posix()
    if rel == '.pbi' or rel.startswith('.pbi/'):
        continue
    if rel == 'definition.pbir':
        pbir = json.loads(file_path.read_text(encoding='utf-8'))
        pbir['datasetReference'] = {'byConnection': {'connectionString': f'semanticmodelid={semantic_model_id}'}}
        raw_bytes = (json.dumps(pbir, indent=2) + '\n').encode('utf-8')
    else:
        raw_bytes = file_path.read_bytes()
    rpt_parts.append({'path': rel, 'payload': base64.b64encode(raw_bytes).decode('ascii'), 'payloadType': 'InlineBase64'})
rpt_payload = {
    'displayName': REPORT_NAME,
    'description': 'Direct Lake analytical report for governed ETL migration assessment portfolios.',
    'definition': {'format': 'PBIR', 'parts': rpt_parts},
}
report = upsert_defined_item(workspace_id, 'reports', REPORT_NAME, rpt_payload, token, folder_id=folder_id)
report_id = report['id']
report_url = f'https://app.fabric.microsoft.com/groups/{workspace_id}/reports/{report_id}'
print(f'Report ready: {REPORT_NAME} ({report_id})')

print(json.dumps({
    'FABRIC_WORKSPACE_ID': workspace_id,
    'FABRIC_FOLDER_ID': folder_id,
    'FABRIC_LAKEHOUSE_ID': lakehouse_id,
    'FABRIC_ENVIRONMENT_ID': environment_id,
    'FABRIC_NOTEBOOK_ID': notebook_id,
    'FABRIC_SEMANTIC_MODEL_ID': semantic_model_id,
    'FABRIC_REPORT_ID': report_id,
    'VITE_REPORT_URL': report_url,
}, indent=2))
